In [ ]:
import logging
from importlib import reload
import os

import torch
import torch.nn as nn
import cv2
import numpy as np
from tqdm import trange

from pathlib import Path

from marmopose.version import __version__ as marmopose_version
from marmopose.config import Config
from marmopose.processing.prediction import Predictor
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import json


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(name)s - %(message)s')
logger = logging.getLogger(__name__)

logger.info(f'MarmoPose version: {marmopose_version}')

cage = 'home'
# cage = 'home'
Cage = cage.capitalize()

n_cams = 4 if cage == 'etho' else 6
cage2 = 'home' if cage == 'etho' else 'etho'

In [ ]:
config_path = '../configs/default.yaml'

# config = Config(
#     config_path=config_path,
    
#     n_tracks=1,
#     project='../demos/test',
#     det_model= '../data/detection_model_finetune_home_onlymarmoset',
#     pose_model= '../data/pose_model_finetune_home',

# )
# print(config.sub_directory)

# config_moreemptyframes = Config(
#     config_path=config_path,
    
#     n_tracks=1,
#     project='../demos/test',
#     det_model= '../data/detection_model_finetune_home_more_empty_frames',
#     pose_model= '../data/pose_model_finetune_home',

# )
# print(config.sub_directory)

config_finetune = Config(
    config_path=config_path,
    
    n_tracks=1,
    project='../demos/test',
    det_model= f'../data/detection_model_finetune_{cage}',
    pose_model= f'../data/pose_model_finetune_{cage}',
    # det_model= f'../data/detection_model_finetune_{cage}_with_{cage2}',
    # pose_model= f'../data/pose_model_finetune_{cage}_with_{cage2}',

)
print(config_finetune.sub_directory)

config_finetune_with = Config(
    config_path=config_path,
    
    n_tracks=1,
    project='../demos/test',
    det_model= f'../data/detection_model_finetune_{cage}_with_{cage2}',
    pose_model= f'../data/pose_model_finetune_{cage}_with_{cage2}',

)
print(config_finetune.sub_directory)


config_base = Config(
    config_path=config_path,
    
    n_tracks=1,
    project='../demos/test',
)
print(config_base.sub_directory)

In [ ]:
predictor = Predictor(config_finetune, batch_size=4)
predictor_base = Predictor(config_base, batch_size=4)
predictor_with = Predictor(config_finetune_with, batch_size=4)

In [ ]:
import json
dataset_dir = f'../../Sleap/TestData{Cage}With/marmoset_family/'
with open(os.path.join(dataset_dir,'annotations/all.json'), 'r') as f:
    test_json = json.load(f)
img_ids = [ann['image_id'] for ann in test_json['annotations']]
# img_ids_annotated = [ann['image_id'] for ann in test_json['annotations'] if 'keypoints' in ann.keys()]
# img_ids_unannotated = list(set(img_ids) - set(img_ids_annotated))
images = [cv2.imread(os.path.join(dataset_dir,'images',img['file_name'])) for img in test_json['images'] if img['id'] in img_ids]

gt_keypoints = []
gt_bboxes = []
for ann in test_json['annotations']:
    if 'keypoints' in ann.keys():
        gt_keypoints.append(ann['keypoints'])
    else:
        gt_keypoints.append([np.nan]*48)

    if 'bbox' in ann.keys():
        gt_bboxes.append(ann['bbox'])
    else:
        gt_bboxes.append([np.nan]*4)

gt_keypoints = np.array(gt_keypoints).reshape((-1,1,16,3))
gt_keypoints[gt_keypoints[:,:,:,2] == 0] = np.nan
gt_bboxes = np.array(gt_bboxes).reshape((-1,1,4))
gt_bboxes[:,:,2] = gt_bboxes[:,:,0] + gt_bboxes[:,:,2]
gt_bboxes[:,:,3] = gt_bboxes[:,:,1] + gt_bboxes[:,:,3]


In [ ]:
sorted_indices = np.load(f'../../Sleap/TestData{Cage}With/sorted_indices.npy')
missing_indices = np.load(f'../../Sleap/TestData{Cage}With/missing_indices.npy')
sorted_images = np.array(images)[sorted_indices]
sorted_gt_keypoints = gt_keypoints[sorted_indices]
sorted_gt_bboxes = gt_bboxes[sorted_indices]


In [ ]:
batch_size = 16
import gc
torch.cuda.empty_cache()
gc.collect()
points_with_score_2d_finetuned = np.empty((0,1,16,3))
bboxes_finetuned = np.empty((0,1,4))
points_with_score_2d_finetuned_with = np.empty((0,1,16,3))
bboxes_finetuned_with = np.empty((0,1,4))
points_with_score_2d_base = np.empty((0,1,16,3))
bboxes_base = np.empty((0,1,4))
for i in range(0,len(sorted_images),batch_size):
    points_with_score_2d_finetuned_part, bboxes_finetuned_part = predictor.predict_image_batch(sorted_images[i:i + batch_size])
    points_with_score_2d_finetuned_with_part, bboxes_finetuned_with_part = predictor_with.predict_image_batch(sorted_images[i:i + batch_size])
    points_with_score_2d_base_part, bboxes_base_part = predictor_base.predict_image_batch(sorted_images[i:i + batch_size])
    points_with_score_2d_finetuned = np.concatenate((points_with_score_2d_finetuned,points_with_score_2d_finetuned_part), axis=0)
    bboxes_finetuned = np.concatenate((bboxes_finetuned,bboxes_finetuned_part), axis=0)
    points_with_score_2d_finetuned_with = np.concatenate((points_with_score_2d_finetuned_with,points_with_score_2d_finetuned_with_part), axis=0)
    bboxes_finetuned_with = np.concatenate((bboxes_finetuned_with,bboxes_finetuned_with_part), axis=0)
    points_with_score_2d_base = np.concatenate((points_with_score_2d_base,points_with_score_2d_base_part), axis=0)
    bboxes_base = np.concatenate((bboxes_base,bboxes_base_part), axis=0)


In [ ]:
def compute_true_false_positives(bboxes):
    frame_ids = np.arange(sorted_gt_bboxes.shape[0])
    frame_ids_absent = np.unique(np.nonzero(np.isnan(sorted_gt_bboxes))[0])
    frame_ids_present = np.setdiff1d(frame_ids, frame_ids_absent, assume_unique=True)
    truepositive = ~np.isnan(bboxes[frame_ids_present,0,0])
    falsepositive = ~np.isnan(bboxes[frame_ids_absent,0,0])
    perc_truepositive = 100 * (np.sum(truepositive)/frame_ids_present.size)
    perc_falsepositive = 100 * (np.sum(falsepositive)/frame_ids_absent.size)
    print(f'False positives: {frame_ids_absent[np.nonzero(falsepositive)]}')
    print(f'False negatives: {frame_ids_present[np.nonzero(~truepositive)]}')
    return perc_truepositive, perc_falsepositive




In [ ]:
print(np.isnan(bboxes_finetuned_with[10:20,0,0]))
print(np.isnan(sorted_gt_bboxes[10:20,0,0]))

In [ ]:
perc_truepositives, perc_falsepositives  = list(zip(*[compute_true_false_positives(bboxes) for bboxes in (bboxes_base, bboxes_finetuned, bboxes_finetuned_with)]))

scores = ['True positives', 'False positives']
models = ['Base', 'Finetuned', f'Finetuned with {cage2}']
width = 1/(len(models) + 1)
offsets = np.arange(width + width/2, 1, width)
xs = np.arange(2)
for i, (tp, fp) in enumerate(zip(perc_truepositives, perc_falsepositives)):
    plt.bar(xs + offsets[i], [tp, fp], width, label = models[i],edgecolor = 'black')
plt.xticks(xs + 0.5 + width/2, labels=scores)
plt.yticks(np.linspace(0,100,5))
plt.ylabel('Percentage')
plt.legend()
plt.show()

In [ ]:
def compute_iou(bboxes):
    frame_ids_present_gt = np.unique(np.nonzero(~np.isnan(sorted_gt_bboxes))[0])
    frame_ids_present = np.unique(np.nonzero(~np.isnan(bboxes))[0])
    frame_ids_present_both = np.intersect1d(frame_ids_present_gt, frame_ids_present, assume_unique=True)
    bboxes_gt_present = sorted_gt_bboxes[frame_ids_present_both,0,:]
    bboxes_present = bboxes[frame_ids_present_both,0,:]
    assert np.sum(np.isnan(bboxes_gt_present)) == 0
    assert np.sum(np.isnan(bboxes_present)) == 0
    assert np.sum(bboxes_present[:,0] < bboxes_present[:,2]) == bboxes_present.shape[0]
    assert np.sum(bboxes_present[:,1] < bboxes_present[:,3]) == bboxes_present.shape[0]
    assert np.sum(bboxes_gt_present[:,0] < bboxes_gt_present[:,2]) == bboxes_gt_present.shape[0]
    assert np.sum(bboxes_gt_present[:,1] < bboxes_gt_present[:,3]) == bboxes_gt_present.shape[0]
    bboxes_inter = np.zeros_like(bboxes_present)
    bboxes_inter[:,:2] = np.max(np.concatenate((bboxes_present[:,:2,None],bboxes_gt_present[:,:2,None]),axis=2),axis=2)
    bboxes_inter[:,2:] = np.min(np.concatenate((bboxes_present[:,2:,None],bboxes_gt_present[:,2:,None]),axis=2),axis=2)
    area_inter = (bboxes_inter[:,2] - bboxes_inter[:,0]) * (bboxes_inter[:,3] - bboxes_inter[:,1])
    area_inter[bboxes_inter[:,0] > bboxes_inter[:,2]] = 0
    area_inter[bboxes_inter[:,1] > bboxes_inter[:,3]] = 0
    area1 = (bboxes_present[:,2] - bboxes_present[:,0]) * (bboxes_present[:,3] - bboxes_present[:,1])
    area2 = (bboxes_gt_present[:,2] - bboxes_gt_present[:,0]) * (bboxes_gt_present[:,3] - bboxes_gt_present[:,1])
    area_union = area1 + area2 - area_inter
    sidx = np.nonzero(area_union < 0)
    return frame_ids_present_both, area_inter/area_union


In [ ]:


idx, ious  = zip(*[compute_iou(bboxes) for bboxes in (bboxes_base, bboxes_finetuned, bboxes_finetuned_with)])
models = ['Base', 'Finetuned', f'Finetuned with {cage2}']
xs = np.arange(1, len(models) + 1)
for i in range(len(models)):
    print(f'{models[i]} IoU: {np.mean(ious[i])}')
    plt.violinplot(ious[i:i+1], xs[i:i+1], showmeans=True)
plt.xticks(xs, labels=models)
plt.ylabel('IoU')
plt.show()
plt.boxplot(ious,showmeans=True,meanline=True)
plt.xticks(xs, labels=models)
plt.ylabel('IoU')
plt.show()

images_idx_low_iou = idx[2][np.argsort(ious[2])[:20]]

In [ ]:
def compute_perc_correct_per_threshold_2D(bboxes, predicted, thresholds):
    frame_ids_present_gt = np.unique(np.nonzero(~np.isnan(sorted_gt_bboxes))[0])
    frame_ids_present = np.unique(np.nonzero(~np.isnan(bboxes))[0])
    frame_ids_present_both = np.intersect1d(frame_ids_present_gt, frame_ids_present, assume_unique=True)

    gt_present = sorted_gt_keypoints[frame_ids_present_both, ...]
    pred_present = predicted[frame_ids_present_both, ...]
    non_labelled_head_gt = np.sum(np.isnan(gt_present[:,:,:3,2]))
    labelled_head_gt = gt_present[:,:,:3,2].size - non_labelled_head_gt
    non_labelled_body_gt = np.sum(np.isnan(gt_present[:,:,[3,8],2]))
    labelled_body_gt = gt_present[:,:,[3,8],2].size - non_labelled_body_gt
    non_labelled_limbs_gt = np.sum(np.isnan(gt_present[:,:,np.r_[4:8,9:13],2]))
    labelled_limbs_gt = gt_present[:,:,np.r_[4:8,9:13],2].size - non_labelled_limbs_gt
    non_labelled_tail_gt = np.sum(np.isnan(gt_present[:,:,13:,2]))
    labelled_tail_gt = gt_present[:,:,13:,2].size - non_labelled_tail_gt
    error_head = (pred_present[:,:,:3,:2] - gt_present[:,:,:3,:2])
    error_head = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_head,error_head))
    perc_head = [100 * np.sum(error_head < thresh)/labelled_head_gt for thresh in thresholds]

    error_body = (pred_present[:,:,[3,8],:2] - gt_present[:,:,[3,8],:2])
    error_body = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_body,error_body))
    perc_body = [100 * np.sum(error_body < thresh)/labelled_body_gt for thresh in thresholds]

    error_limbs = (pred_present[:,:,np.r_[4:8,9:13],:2] - gt_present[:,:,np.r_[4:8,9:13],:2])
    error_limbs = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_limbs,error_limbs))
    perc_limbs = [100 * np.sum(error_limbs < thresh)/labelled_limbs_gt for thresh in thresholds]

    error_tail = (pred_present[:,:,13:,:2] - gt_present[:,:,13:,:2])
    error_tail = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_tail,error_tail))
    perc_tail = [100 * np.sum(error_tail < thresh)/labelled_tail_gt for thresh in thresholds]

    return perc_head, perc_body, perc_limbs, perc_tail


In [ ]:
example = 1

thresholds = np.arange(0,70,2)
allbboxes = [bboxes_base, bboxes_finetuned, bboxes_finetuned_with]
allpoints = [points_with_score_2d_base, points_with_score_2d_finetuned, points_with_score_2d_finetuned_with]

colors = ['r', 'b','purple']
for i in range(len(models)):
    perc_head, perc_body, perc_limbs, perc_tail = compute_perc_correct_per_threshold_2D(allbboxes[i], allpoints[i], thresholds)
    plt.plot(thresholds,perc_head,c=colors[i],marker = 'o',ms=4, label = models[i] + ' head')
    plt.plot(thresholds,perc_body,c=colors[i],marker = 's',ms=4, label = models[i] + ' body')
    plt.plot(thresholds,perc_limbs,c=colors[i],marker = '^',ms=4, label = models[i] + ' limbs')
    plt.plot(thresholds,perc_tail,c=colors[i],marker = 'd',ms=4, label = models[i] + ' tail')
plt.legend()
plt.xlabel('Error threshold (pixels)')
plt.ylabel('Accuracy (%)')
plt.ylim((0,100))
plt.show()


# perc_head_finetuned, perc_body_finetuned, perc_limbs_finetuned, perc_tail_finetuned = compute_perc_correct_per_threshold_2D(sorted_gt_keypoints, points_with_score_2d_finetuned, thresholds)
# perc_head_base, perc_body_base, perc_limbs_base, perc_tail_base = compute_perc_correct_per_threshold_2D(sorted_gt_keypoints, points_with_score_2d_base, thresholds)
# plt.plot(thresholds,perc_head_finetuned,c='b',marker = 'o',ms=4)
# plt.plot(thresholds,perc_body_finetuned,c='b',marker = 's',ms=4)
# plt.plot(thresholds,perc_limbs_finetuned,c='b',marker = '^',ms=4)
# plt.plot(thresholds,perc_tail_finetuned,c='b',marker = 'd',ms=4)

# plt.plot(thresholds,perc_head_base,c='r',marker = 'o',ms=4)
# plt.plot(thresholds,perc_body_base,c='r',marker = 's',ms=4)
# plt.plot(thresholds,perc_limbs_base,c='r',marker = '^',ms=4)
# plt.plot(thresholds,perc_tail_base,c='r',marker = 'd',ms=4)

# n_ns = len(points_with_score_2d_dict.keys())
# for i, n in enumerate(sorted(points_with_score_2d_dict.keys())):
#     perc_head, perc_body, perc_limbs, perc_tail = compute_perc_correct_per_threshold_2D(sorted_gt_keypoints, points_with_score_2d_dict[n], thresholds)
#     plt.plot(thresholds,perc_head,c=((n_ns - i)/(n_ns + 1),(n_ns - i)/(n_ns + 1),1),marker = 'o',ms=4)
#     plt.plot(thresholds,perc_body,c=((n_ns - i)/(n_ns + 1),(n_ns - i)/(n_ns + 1),1),marker = 's',ms=4)
#     plt.plot(thresholds,perc_limbs,c=((n_ns - i)/(n_ns + 1),(n_ns - i)/(n_ns + 1),1),marker = '^',ms=4)
#     plt.plot(thresholds,perc_tail,c=((n_ns - i)/(n_ns + 1),(n_ns - i)/(n_ns + 1),1),marker = 'd',ms=4)



# plt.xlabel('Error threshold (pixels)')
# plt.ylabel('Accuracy (%)')
# plt.ylim((0,100))
# plt.show()

In [ ]:
bboxes_1 = bboxes_finetuned
points_with_score_2d_1 = points_with_score_2d_finetuned
bboxes_2 = bboxes_finetuned_with
points_with_score_2d_2 = points_with_score_2d_finetuned_with
jitter = 20
for i, image in zip(images_idx_low_iou, sorted_images[images_idx_low_iou]):
# for i, image in enumerate(sorted_images[jitter:jitter+20]):
#     i = i + jitter
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(20, 10))
    axes[0].set_title(img_ids[i])
    axes[0].imshow(image[:,:,[2,1,0]])
    axes[1].imshow(image[:,:,[2,1,0]])
    rect = patches.Rectangle(
        (sorted_gt_bboxes[i,0,0], sorted_gt_bboxes[i,0,1]), sorted_gt_bboxes[i,0,2] - sorted_gt_bboxes[i,0,0], sorted_gt_bboxes[i,0,3] - sorted_gt_bboxes[i,0,1],
        linewidth=1, edgecolor='g', facecolor='none'
    )
    axes[0].add_patch(rect)
    axes[0].scatter(sorted_gt_keypoints[i,0,:,0],sorted_gt_keypoints[i,0,:,1],c='g',s = 1)
    nans1 = np.isnan(points_with_score_2d_1[i,0,:,2])
    if not np.sum(nans1) == 16:
        axes[0].scatter(points_with_score_2d_1[i,0,~nans1,0],points_with_score_2d_1[i,0,~nans1,1],c='r',s = points_with_score_2d_1[i,0,~nans1,2])
    rect = patches.Rectangle(
        (sorted_gt_bboxes[i,0,0], sorted_gt_bboxes[i,0,1]), sorted_gt_bboxes[i,0,2] - sorted_gt_bboxes[i,0,0], sorted_gt_bboxes[i,0,3] - sorted_gt_bboxes[i,0,1],
        linewidth=1, edgecolor='g', facecolor='none'
    )
    
    axes[1].add_patch(rect)
    axes[1].scatter(sorted_gt_keypoints[i,0,:,0],sorted_gt_keypoints[i,0,:,1],c='g',s = 1)
    nans2 = np.isnan(points_with_score_2d_2[i,0,:,2])
    axes[1].scatter(points_with_score_2d_2[i,0,~nans2,0],points_with_score_2d_2[i,0,~nans2,1],c='b',s = points_with_score_2d_2[i,0,~nans2,2])
    rect = patches.Rectangle(
        (bboxes_1[i,0,0], bboxes_1[i,0,1]), bboxes_1[i,0,2]-bboxes_1[i,0,0], bboxes_1[i,0,3]-bboxes_1[i,0,1],
        linewidth=1, edgecolor='r', facecolor='none'
    )
    axes[0].add_patch(rect)
    rect = patches.Rectangle(
        (bboxes_2[i,0,0], bboxes_2[i,0,1]), bboxes_2[i,0,2]-bboxes_2[i,0,0], bboxes_2[i,0,3]-bboxes_2[i,0,1],
        linewidth=1, edgecolor='b', facecolor='none'
    )
    axes[1].add_patch(rect)

    for bodyparts in config_base.visualization['skeleton'][::-1]:
        idx_bodyparts = []
        for bodypart in bodyparts:
            idx_bodyparts.append(config_base.animal['bodyparts'].index(bodypart))
        axes[0].plot(sorted_gt_keypoints[i,0,idx_bodyparts,0],sorted_gt_keypoints[i,0,idx_bodyparts,1], lw=1,color='g')
        axes[1].plot(sorted_gt_keypoints[i,0,idx_bodyparts,0],sorted_gt_keypoints[i,0,idx_bodyparts,1], lw=1,color='g')
        axes[0].plot(points_with_score_2d_1[i,0,idx_bodyparts,0],points_with_score_2d_1[i,0,idx_bodyparts,1], lw=1,color='r')
        axes[1].plot(points_with_score_2d_2[i,0,idx_bodyparts,0],points_with_score_2d_2[i,0,idx_bodyparts,1], lw=1,color='b')

    axes[0].axis('off')
    axes[1].axis('off')
    fig.show()




In [ ]:
# start_indices = np.concatenate((np.array([0]), 1 + missing_indices))
# end_indices = np.concatenate((missing_indices,np.array([sorted_indices.size + missing_indices.size])))

# sorted_images_with_missing = np.full((sorted_indices.size + missing_indices.size, *sorted_images.shape[1:]), 255)
# sorted_gt_keypoints_with_missing = np.full((sorted_indices.size + missing_indices.size, *gt_keypoints.shape[1:]), np.nan)
# sorted_base_keypoints_with_missing = np.full((sorted_indices.size + missing_indices.size, *points_with_score_2d_base.shape[1:]), np.nan)
# sorted_finetuned_keypoints_with_missing = np.full((sorted_indices.size + missing_indices.size, *points_with_score_2d_finetuned.shape[1:]), np.nan)
# for i, (idx0, idx1) in enumerate(zip(start_indices,end_indices)):
#     print(i,idx0,idx1)
#     sorted_images_with_missing[idx0:idx1, ...] = sorted_images[idx0-i:idx1-i, ...]
#     sorted_gt_keypoints_with_missing[idx0:idx1, ...] = sorted_gt_keypoints[idx0 -i:idx1 - i, ...]
#     sorted_base_keypoints_with_missing[idx0:idx1, ...] = points_with_score_2d_base[idx0 -i:idx1 - i, ...]
#     sorted_finetuned_keypoints_with_missing[idx0:idx1, ...] = points_with_score_2d_finetuned[idx0 -i:idx1 - i, ...]

# sorted_images_with_missing = sorted_images_with_missing.reshape(4,-1,*sorted_images_with_missing.shape[1:])
# sorted_gt_keypoints_with_missing = sorted_gt_keypoints_with_missing.reshape(4,-1,*sorted_gt_keypoints_with_missing.shape[2:])
# sorted_gt_keypoints_with_missing[sorted_gt_keypoints_with_missing[:,:,:,2] == 0,:] = np.nan
# print(sorted_gt_keypoints_with_missing.shape)
# sorted_base_keypoints_with_missing = sorted_base_keypoints_with_missing.reshape(4,-1,*sorted_base_keypoints_with_missing.shape[2:])
# sorted_finetuned_keypoints_with_missing = sorted_finetuned_keypoints_with_missing.reshape(4,-1,*sorted_finetuned_keypoints_with_missing.shape[2:])

In [ ]:
from tqdm import trange
from marmopose.calibration.cameras import CameraGroup

if cage == 'etho':
    camera_groups = CameraGroup.load_from_json(os.path.join('/srv','MarmOT','VideoTracking','Videos','CalibEtho','camera_params.json'))

elif cage == 'home':
    camera_groups = [CameraGroup.load_from_json(os.path.join('/srv','MarmOT','VideoTracking','Videos',f'TestHomeWithEtho{i}.1','Calib_preprocessed','camera_params.json')) for i in range(1,5)]
    
def triangulate_frame(camera_group, points_with_score_2d: np.ndarray, ransac=True):
    """
    Args:
        camera_group: CameraGroup
        points_with_score_2d: (n_cams, n_tracks, n_bodyparts, (x, y, score))
    
    Returns:
        points_3d: (n_bodyparts, (x, y, z))
    """


    if ransac:
        points_3d = camera_group.triangulate_ransac(points_with_score_2d, undistort=True)
    else:
        points_3d = camera_group.triangulate(points_with_score_2d, undistort=True)
        
    return points_3d

n_frames_per_cam = int(sorted_indices.size/n_cams)
diff_indices_cam1 = sorted_indices[1:n_frames_per_cam] -  sorted_indices[:n_frames_per_cam - 1]
sessions_for_frames = np.zeros((n_frames_per_cam), dtype = np.uint8)
for i, idx in enumerate(np.nonzero(diff_indices_cam1 != 1)[0] + 1):
    sessions_for_frames[idx:] = i + 1

points_with_score_2d_gt_reshaped = sorted_gt_keypoints.reshape(n_cams, -1, *sorted_gt_keypoints.shape[2:])
points_with_score_2d_base_reshaped = points_with_score_2d_base.reshape(n_cams, -1, *points_with_score_2d_base.shape[2:])
points_with_score_2d_finetuned_reshaped = points_with_score_2d_finetuned.reshape(n_cams, -1, *points_with_score_2d_finetuned.shape[2:])
points_with_score_2d_finetuned_with_reshaped = points_with_score_2d_finetuned_with.reshape(n_cams, -1, *points_with_score_2d_finetuned_with.shape[2:])

n_cams, n_frames, n_bodyparts, n_dim = points_with_score_2d_gt_reshaped.shape
gt_points_3d = np.full((n_frames, n_bodyparts, 3), np.nan)
base_points_3d = np.full((n_frames, n_bodyparts, 3), np.nan)
finetuned_points_3d = np.full((n_frames, n_bodyparts, 3), np.nan)
finetuned_with_points_3d = np.full((n_frames, n_bodyparts, 3), np.nan)

for frame_idx in trange(n_frames, ncols=100, desc='Triangulating... ', unit='frames'):
    gt_all_points_with_score_2d_frame = points_with_score_2d_gt_reshaped[:, frame_idx]
    base_all_points_with_score_2d_frame = points_with_score_2d_base_reshaped[:, frame_idx]
    finetuned_all_points_with_score_2d_frame = points_with_score_2d_finetuned_reshaped[:, frame_idx]
    finetuned_with_all_points_with_score_2d_frame = points_with_score_2d_finetuned_with_reshaped[:, frame_idx]
    if isinstance(camera_groups, list):
        camera_group = camera_groups[sessions_for_frames[frame_idx]]
    else:
        camera_group = camera_groups
    
    gt_point_3d = triangulate_frame(camera_group, gt_all_points_with_score_2d_frame, ransac=True) 
    base_point_3d = triangulate_frame(camera_group, base_all_points_with_score_2d_frame, ransac=True) 
    finetuned_point_3d = triangulate_frame(camera_group, finetuned_all_points_with_score_2d_frame, ransac=True) 
    finetuned_with_point_3d = triangulate_frame(camera_group, finetuned_with_all_points_with_score_2d_frame, ransac=True) 
        
    gt_points_3d[frame_idx] = gt_point_3d
    base_points_3d[frame_idx] = base_point_3d
    finetuned_points_3d[frame_idx] = finetuned_point_3d
    finetuned_with_points_3d[frame_idx] = finetuned_with_point_3d



In [ ]:
def compute_perc_correct_per_threshold_3D(groundtruth, predicted, thresholds):
    gt_head = groundtruth[:,:3,:]
    non_labelled_head_gt = np.sum(np.isnan(gt_head[:,:,2]))
    labelled_head_gt = gt_head[:,:,2].size - non_labelled_head_gt
    gt_body = groundtruth[:,[3,8],:]
    non_labelled_body_gt = np.sum(np.isnan(gt_body[:,:,2]))
    labelled_body_gt = gt_body[:,:,2].size - non_labelled_body_gt
    gt_limbs = groundtruth[:,np.r_[4:8,9:13],:]
    non_labelled_limbs_gt = np.sum(np.isnan(gt_limbs[:,:,2]))
    labelled_limbs_gt = gt_limbs[:,:,2].size - non_labelled_limbs_gt
    gt_tail = groundtruth[:,13:,:]
    non_labelled_tail_gt = np.sum(np.isnan(gt_tail[:,:,2]))
    labelled_tail_gt = gt_tail[:,:,2].size - non_labelled_tail_gt

    error_head_finetuned = (predicted[:,:3,:] - gt_head)
    error_head_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_head_finetuned,error_head_finetuned))
    perc_head_finetuned = [100 * np.sum(error_head_finetuned < thresh)/labelled_head_gt for thresh in thresholds]

    error_body_finetuned = (predicted[:,[3,8],:] - gt_body)
    error_body_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_body_finetuned,error_body_finetuned))
    perc_body_finetuned = [100 * np.sum(error_body_finetuned < thresh)/labelled_body_gt for thresh in thresholds]

    error_limbs_finetuned = (predicted[:,np.r_[4:8,9:13],:] - gt_limbs)
    error_limbs_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_limbs_finetuned,error_limbs_finetuned))
    perc_limbs_finetuned = [100 * np.sum(error_limbs_finetuned < thresh)/labelled_limbs_gt for thresh in thresholds]

    error_tail_finetuned = (predicted[:,13:,:] - gt_tail)
    error_tail_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_tail_finetuned,error_tail_finetuned))
    perc_tail_finetuned = [100 * np.sum(error_tail_finetuned < thresh)/labelled_tail_gt for thresh in thresholds]

    return perc_head_finetuned, perc_body_finetuned, perc_limbs_finetuned, perc_tail_finetuned


In [ ]:
thresholds = np.arange(0,101,4)

perc_head_finetuned, perc_body_finetuned, perc_limbs_finetuned, perc_tail_finetuned = compute_perc_correct_per_threshold_3D(gt_points_3d, finetuned_points_3d, thresholds)
perc_head_finetuned_with, perc_body_finetuned_with, perc_limbs_finetuned_with, perc_tail_finetuned_with = compute_perc_correct_per_threshold_3D(gt_points_3d, finetuned_with_points_3d, thresholds)
perc_head_base, perc_body_base, perc_limbs_base, perc_tail_base = compute_perc_correct_per_threshold_3D(gt_points_3d, base_points_3d, thresholds)
plt.plot(thresholds,perc_head_finetuned,c='b',marker = 'o',ms=4)
plt.plot(thresholds,perc_body_finetuned,c='b',marker = 's',ms=4)
plt.plot(thresholds,perc_limbs_finetuned,c='b',marker = '^',ms=4)
plt.plot(thresholds,perc_tail_finetuned,c='b',marker = 'd',ms=4)

plt.plot(thresholds,perc_head_finetuned_with,c='purple',marker = 'o',ms=4)
plt.plot(thresholds,perc_body_finetuned_with,c='purple',marker = 's',ms=4)
plt.plot(thresholds,perc_limbs_finetuned_with,c='purple',marker = '^',ms=4)
plt.plot(thresholds,perc_tail_finetuned_with,c='purple',marker = 'd',ms=4)

plt.plot(thresholds,perc_head_base,c='r',marker = 'o',ms=4)
plt.plot(thresholds,perc_body_base,c='r',marker = 's',ms=4)
plt.plot(thresholds,perc_limbs_base,c='r',marker = '^',ms=4)
plt.plot(thresholds,perc_tail_base,c='r',marker = 'd',ms=4)
plt.xlabel('Error threshold (mm)')
plt.ylabel('Accuracy (%)')
plt.ylim((0,100))
plt.show()

In [ ]:
xlim, ylim, zlim, _ = (1200, 730, 900, 30) if cage == 'home' else (660, 560, 800, 30)

for i in range(0,30):
    # Create a 3D scatter plot
    fig = plt.figure(figsize=(10,5))
    ax1 = fig.add_subplot(131, projection='3d')
    ax2 = fig.add_subplot(132, projection='3d')
    ax3 = fig.add_subplot(133, projection='3d')
    for bodyparts in config_base.visualization['skeleton'][::-1]:
        idx_bodyparts = []
        for bodypart in bodyparts:
            idx_bodyparts.append(config_base.animal['bodyparts'].index(bodypart))
        ax1.plot(gt_points_3d[i,idx_bodyparts,0],gt_points_3d[i,idx_bodyparts,1],gt_points_3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='g')
        ax2.plot(base_points_3d[i,idx_bodyparts,0],base_points_3d[i,idx_bodyparts,1],base_points_3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='r')
        ax3.plot(finetuned_points_3d[i,idx_bodyparts,0],finetuned_points_3d[i,idx_bodyparts,1],finetuned_points_3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='b')
    ax1.set_title(f'frame {i} ground truth')
    ax1.set_xlim((0,xlim))
    ax1.set_ylim((0,ylim))
    ax1.set_zlim((0,zlim))
    ax2.set_title(f'frame {i} base model')
    ax2.set_xlim((0,xlim))
    ax2.set_ylim((0,ylim))
    ax2.set_zlim((0,zlim))
    ax3.set_title(f'frame {i} finetuned model')
    ax3.set_xlim((0,xlim))
    ax3.set_ylim((0,ylim))
    ax3.set_zlim((0,zlim))

In [ ]:
from marmopose.utils.data_io import load_points_3d_h5
with open(f'../../Sleap/TestData{Cage}With/frames_dict.json') as f:
    frames_in_videos = json.load(f)
print(frames_in_videos)
fullworkflow_finetuned_points3d = np.full((finetuned_points_3d.shape), np.nan)
fullworkflow_finetuned_with_points3d = np.full((finetuned_with_points_3d.shape), np.nan)
fullworkflow_base_points3d = np.full((finetuned_points_3d.shape), np.nan)
i = 0
for video in sorted(frames_in_videos.keys()):
    points_3d = load_points_3d_h5(f"../../Videos/Test{Cage}With{Cage2}{int(video)}.1/Output/points_3d/optimized.h5")
    fullworkflow_finetuned_points3d[i:i+len(frames_in_videos[video]),:,:] = points_3d[0,frames_in_videos[video],:,:]
    points_3d = load_points_3d_h5(f"../../Videos/Test{Cage}With{Cage2}{int(video)}.1/Output_basemodel/points_3d/optimized.h5")
    fullworkflow_base_points3d[i:i+len(frames_in_videos[video]),:,:] = points_3d[0,frames_in_videos[video],:,:]
    points_3d = load_points_3d_h5(f"../../Videos/Test{Cage}With{Cage2}{int(video)}.1/Output_basemodel/points_3d/optimized.h5")
    fullworkflow_base_points3d[i:i+len(frames_in_videos[video]),:,:] = points_3d[0,frames_in_videos[video],:,:]
    i += len(frames_in_videos[video])


In [ ]:
thresholds = np.arange(0,70,2)
perc_head_fullworkflow_finetuned, perc_body_fullworkflow_finetuned, perc_limbs_fullworkflow_finetuned, perc_tail_fullworkflow_finetuned = compute_perc_correct_per_threshold_3D(gt_points_3d, fullworkflow_finetuned_points3d, thresholds)
perc_head_fullworkflow_base, perc_body_fullworkflow_base, perc_limbs_fullworkflow_base, perc_tail_fullworkflow_base = compute_perc_correct_per_threshold_3D(gt_points_3d, fullworkflow_base_points3d, thresholds)


plt.plot(thresholds,perc_head_fullworkflow_finetuned,c='b',marker = 'o',ms=4)
plt.plot(thresholds,perc_body_fullworkflow_finetuned,c='b',marker = 's',ms=4,)
plt.plot(thresholds,perc_limbs_fullworkflow_finetuned,c='b',marker = '^',ms=4)
plt.plot(thresholds,perc_tail_fullworkflow_finetuned,c='b',marker = 'd',ms=4)

# plt.plot(thresholds,perc_head_fullworkflow_base,c='r',marker = 'o',ms=4)
# plt.plot(thresholds,perc_body_fullworkflow_base,c='r',marker = 's',ms=4,)
# plt.plot(thresholds,perc_limbs_fullworkflow_base,c='r',marker = '^',ms=4)
# plt.plot(thresholds,perc_tail_fullworkflow_base,c='r',marker = 'd',ms=4)

plt.xlabel('Error threshold (mm)')
plt.ylabel('Accuracy (%)')
plt.ylim((0,100))
plt.show()

In [ ]:
xlim, ylim, zlim, _ = config.visualization['room_dimensions']

for i in range(30):
    # Create a 3D scatter plot
    fig = plt.figure(figsize=(10,5))
    ax1 = fig.add_subplot(131, projection='3d')
    ax2 = fig.add_subplot(132, projection='3d')
    ax3 = fig.add_subplot(133, projection='3d')
    for bodyparts in config.visualization['skeleton'][::-1]:
        idx_bodyparts = []
        for bodypart in bodyparts:
            idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
        ax1.plot(gt_points_3d[i,idx_bodyparts,0],gt_points_3d[i,idx_bodyparts,1],gt_points_3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='g')
        ax2.plot(finetuned_points_3d[i,idx_bodyparts,0],finetuned_points_3d[i,idx_bodyparts,1],finetuned_points_3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='b')
        ax3.plot(fullworkflow_finetuned_points3d[i,idx_bodyparts,0],fullworkflow_finetuned_points3d[i,idx_bodyparts,1],fullworkflow_finetuned_points3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='indigo')
    ax1.set_title(f'frame {i} ground truth')
    ax1.set_xlim((0,xlim))
    ax1.set_ylim((0,ylim))
    ax1.set_zlim((0,zlim))
    ax2.set_title(f'frame {i} finetuned model')
    ax2.set_xlim((0,xlim))
    ax2.set_ylim((0,ylim))
    ax2.set_zlim((0,zlim))
    ax3.set_title(f'frame {i} fullworkflow finetuned model')
    ax3.set_xlim((0,xlim))
    ax3.set_ylim((0,ylim))
    ax3.set_zlim((0,zlim))

In [ ]:
import logging
from importlib import reload
import os

import torch
import torch.nn as nn
import cv2
import numpy as np
from tqdm import trange

from pathlib import Path

from marmopose.version import __version__ as marmopose_version
from marmopose.config import Config
from marmopose.processing.prediction import Predictor
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import json

config_path = '../configs/default.yaml'

config_finetune_etho_with = Config(
    config_path=config_path,
    
    n_tracks=1,
    project='../demos/test',
    det_model= f'../data/detection_model_finetune_etho_with_home',
    pose_model= f'../data/pose_model_finetune_etho_with_home',
)

config_finetune_home_with = Config(
    config_path=config_path,
    
    n_tracks=1,
    project='../demos/test',
    det_model= f'../data/detection_model_finetune_home_with_etho',
    pose_model= f'../data/pose_model_finetune_home_with_etho',
)

predictor_etho = Predictor(config_finetune_etho_with, batch_size=4)
predictor_home = Predictor(config_finetune_home_with, batch_size=4)


In [ ]:
import json
dataset_dir_home = f'../../Sleap/TestDataHomeWith/marmoset_family/'
with open(os.path.join(dataset_dir_home,'annotations/all.json'), 'r') as f:
    test_json = json.load(f)
img_ids = [ann['image_id'] for ann in test_json['annotations']]
# img_ids_annotated = [ann['image_id'] for ann in test_json['annotations'] if 'keypoints' in ann.keys()]
# img_ids_unannotated = list(set(img_ids) - set(img_ids_annotated))
images = [cv2.imread(os.path.join(dataset_dir_home,'images',img['file_name'])) for img in test_json['images'] if img['id'] in img_ids]

gt_keypoints_home = []
gt_bboxes_home = []
for ann in test_json['annotations']:
    if 'keypoints' in ann.keys():
        gt_keypoints_home.append(ann['keypoints'])
    else:
        gt_keypoints_home.append([np.nan]*48)

    if 'bbox' in ann.keys():
        gt_bboxes_home.append(ann['bbox'])
    else:
        gt_bboxes_home.append([np.nan]*4)

gt_keypoints_home = np.array(gt_keypoints_home).reshape((-1,1,16,3))
gt_keypoints_home[gt_keypoints_home[:,:,:,2] == 0] = np.nan
gt_bboxes_home = np.array(gt_bboxes_home).reshape((-1,1,4))
gt_bboxes_home[:,:,2] = gt_bboxes_home[:,:,0] + gt_bboxes_home[:,:,2]
gt_bboxes_home[:,:,3] = gt_bboxes_home[:,:,1] + gt_bboxes_home[:,:,3]

sorted_indices_home = np.load(f'../../Sleap/TestDataHomeWith/sorted_indices.npy')
missing_indices = np.load(f'../../Sleap/TestDataHomeWith/missing_indices.npy')
sorted_images_home = np.array(images)[sorted_indices_home]
sorted_gt_keypoints_home = gt_keypoints_home[sorted_indices_home]
sorted_gt_bboxes_home = gt_bboxes_home[sorted_indices_home]

dataset_dir_etho = f'../../Sleap/TestDataEthoWith/marmoset_family/'
with open(os.path.join(dataset_dir_etho,'annotations/all.json'), 'r') as f:
    test_json = json.load(f)
img_ids = [ann['image_id'] for ann in test_json['annotations']]
# img_ids_annotated = [ann['image_id'] for ann in test_json['annotations'] if 'keypoints' in ann.keys()]
# img_ids_unannotated = list(set(img_ids) - set(img_ids_annotated))
images = [cv2.imread(os.path.join(dataset_dir_etho,'images',img['file_name'])) for img in test_json['images'] if img['id'] in img_ids]

gt_keypoints_etho = []
gt_bboxes_etho = []
for ann in test_json['annotations']:
    if 'keypoints' in ann.keys():
        gt_keypoints_etho.append(ann['keypoints'])
    else:
        gt_keypoints_etho.append([np.nan]*48)

    if 'bbox' in ann.keys():
        gt_bboxes_etho.append(ann['bbox'])
    else:
        gt_bboxes_etho.append([np.nan]*4)

gt_keypoints_etho = np.array(gt_keypoints_etho).reshape((-1,1,16,3))
gt_keypoints_etho[gt_keypoints_etho[:,:,:,2] == 0] = np.nan
gt_bboxes_etho = np.array(gt_bboxes_etho).reshape((-1,1,4))
gt_bboxes_etho[:,:,2] = gt_bboxes_etho[:,:,0] + gt_bboxes_etho[:,:,2]
gt_bboxes_etho[:,:,3] = gt_bboxes_etho[:,:,1] + gt_bboxes_etho[:,:,3]

sorted_indices_etho = np.load(f'../../Sleap/TestDataEthoWith/sorted_indices.npy')
missing_indices = np.load(f'../../Sleap/TestDataEthoWith/missing_indices.npy')
sorted_images_etho = np.array(images)[sorted_indices_etho]
sorted_gt_keypoints_etho = gt_keypoints_etho[sorted_indices_etho]
sorted_gt_bboxes_etho = gt_bboxes_etho[sorted_indices_etho]

batch_size = 16
import gc
torch.cuda.empty_cache()
gc.collect()
points_with_score_2d_home = np.empty((0,1,16,3))
bboxes_home = np.empty((0,1,4))
points_with_score_2d_etho = np.empty((0,1,16,3))
bboxes_etho = np.empty((0,1,4))
for i in range(0,len(sorted_images_etho),batch_size):
    points_with_score_2d_etho_part, bboxes_etho_part = predictor_etho.predict_image_batch(sorted_images_etho[i:i + batch_size])
    points_with_score_2d_etho = np.concatenate((points_with_score_2d_etho,points_with_score_2d_etho_part), axis=0)
    bboxes_etho = np.concatenate((bboxes_etho,bboxes_etho_part), axis=0)
for i in range(0,len(sorted_images_home),batch_size):
    points_with_score_2d_home_part, bboxes_home_part = predictor_home.predict_image_batch(sorted_images_home[i:i + batch_size])
    points_with_score_2d_home = np.concatenate((points_with_score_2d_home,points_with_score_2d_home_part), axis=0)
    bboxes_home = np.concatenate((bboxes_home,bboxes_home_part), axis=0)


In [ ]:
from tqdm import trange
from marmopose.calibration.cameras import CameraGroup

camera_groups_etho = CameraGroup.load_from_json(os.path.join('/srv','MarmOT','VideoTracking','Videos','CalibEtho','camera_params.json'))

camera_groups_home = [CameraGroup.load_from_json(os.path.join('/srv','MarmOT','VideoTracking','Videos',f'TestHomeWithEtho{i}.1','Calib_preprocessed','camera_params.json')) for i in range(1,5)]

    
def triangulate_frame(camera_group, points_with_score_2d: np.ndarray, ransac=True):
    """
    Args:
        camera_group: CameraGroup
        points_with_score_2d: (n_cams, n_tracks, n_bodyparts, (x, y, score))
    
    Returns:
        points_3d: (n_bodyparts, (x, y, z))
    """


    if ransac:
        points_3d = camera_group.triangulate_ransac(points_with_score_2d, undistort=True)
    else:
        points_3d = camera_group.triangulate(points_with_score_2d, undistort=True)
        
    return points_3d

n_frames_per_cam_home = int(sorted_indices_home.size/4)
diff_indices_cam1 = sorted_indices_home[1:n_frames_per_cam_home] -  sorted_indices_home[:n_frames_per_cam_home - 1]
sessions_for_frames_home = np.zeros((n_frames_per_cam_home), dtype = np.uint8)
for i, idx in enumerate(np.nonzero(diff_indices_cam1 != 1)[0] + 1):
    sessions_for_frames_home[idx:] = i + 1

points_with_score_2d_gt_home_reshaped = sorted_gt_keypoints_home.reshape(6, -1, *sorted_gt_keypoints_home.shape[2:])
points_with_score_2d_home_reshaped = points_with_score_2d_home.reshape(6, -1, *points_with_score_2d_home.shape[2:])

n_cams, n_frames, n_bodyparts, n_dim = points_with_score_2d_gt_home_reshaped.shape
gt_home_points_3d = np.full((n_frames, n_bodyparts, 3), np.nan)
home_points_3d = np.full((n_frames, n_bodyparts, 3), np.nan)
for frame_idx in trange(n_frames, ncols=100, desc='Triangulating... ', unit='frames'):
    gt_home_all_points_with_score_2d_frame = points_with_score_2d_gt_home_reshaped[:, frame_idx]
    home_all_points_with_score_2d_frame = points_with_score_2d_home_reshaped[:, frame_idx]
    if isinstance(camera_groups_home, list):
        print(frame_idx)
        print(sessions_for_frames_home[frame_idx])
        camera_group = camera_groups_home[sessions_for_frames_home[frame_idx]]
    else:
        camera_group = camera_groups_home
    
    gt_home_point_3d = triangulate_frame(camera_group, gt_home_all_points_with_score_2d_frame, ransac=True) 
    home_point_3d = triangulate_frame(camera_group, home_all_points_with_score_2d_frame, ransac=True) 
        
    gt_home_points_3d[frame_idx] = gt_home_point_3d
    home_points_3d[frame_idx] = home_point_3d


n_frames_per_cam_etho = int(sorted_indices_etho.size/6)
diff_indices_cam1 = sorted_indices_etho[1:n_frames_per_cam_etho] -  sorted_indices_etho[:n_frames_per_cam_etho - 1]
sessions_for_frames_etho = np.zeros((n_frames_per_cam_etho), dtype = np.uint8)
for i, idx in enumerate(np.nonzero(diff_indices_cam1 != 1)[0] + 1):
    sessions_for_frames_etho[idx:] = i + 1

points_with_score_2d_gt_etho_reshaped = sorted_gt_keypoints_etho.reshape(4, -1, *sorted_gt_keypoints_etho.shape[2:])
points_with_score_2d_etho_reshaped = points_with_score_2d_etho.reshape(4, -1, *points_with_score_2d_etho.shape[2:])

n_cams, n_frames, n_bodyparts, n_dim = points_with_score_2d_gt_etho_reshaped.shape
gt_etho_points_3d = np.full((n_frames, n_bodyparts, 3), np.nan)
etho_points_3d = np.full((n_frames, n_bodyparts, 3), np.nan)

for frame_idx in trange(n_frames, ncols=100, desc='Triangulating... ', unit='frames'):
    gt_etho_all_points_with_score_2d_frame = points_with_score_2d_gt_etho_reshaped[:, frame_idx]
    etho_all_points_with_score_2d_frame = points_with_score_2d_etho_reshaped[:, frame_idx]
    if isinstance(camera_groups_etho, list):
        camera_group = camera_groups_etho[sessions_for_frames_etho[frame_idx]]
    else:
        camera_group = camera_groups_etho
    
    gt_etho_point_3d = triangulate_frame(camera_group, gt_etho_all_points_with_score_2d_frame, ransac=True) 
    etho_point_3d = triangulate_frame(camera_group, etho_all_points_with_score_2d_frame, ransac=True) 
        
    gt_etho_points_3d[frame_idx] = gt_etho_point_3d
    etho_points_3d[frame_idx] = etho_point_3d




In [ ]:
print(np.isnan(bboxes_finetuned_with[10:20,0,0]))
print(np.isnan(sorted_gt_bboxes[10:20,0,0]))
print(np.isnan(bboxes_etho[10:20,0,0]))
print(np.isnan(sorted_gt_bboxes[10:20,0,0]))


In [ ]:
fs = 26
lfs = 22
tfs = 22
lgfs = 22
lw = 3
color_etho = '#984ea3'
color_home = '#ff7f00'
colors= [color_etho,color_home]
def compute_true_false_positives(bboxes, bboxes_gt):
    print(bboxes.shape)
    frame_ids = np.arange(bboxes_gt.shape[0])
    frame_ids_absent = np.unique(np.nonzero(np.isnan(bboxes_gt))[0])
    frame_ids_present = np.setdiff1d(frame_ids, frame_ids_absent, assume_unique=True)
    truepositive = ~np.isnan(bboxes[frame_ids_present,0,0])
    falsepositive = ~np.isnan(bboxes[frame_ids_absent,0,0])
    perc_truepositive = 100 * (np.sum(truepositive)/frame_ids_present.size)
    perc_falsepositive = 100 * (np.sum(falsepositive)/frame_ids_absent.size)
    print(f'False positives: {frame_ids_absent[np.nonzero(falsepositive)]}')
    print(f'False negatives: {frame_ids_present[np.nonzero(~truepositive)]}')
    return perc_truepositive, perc_falsepositive

def compute_iou(bboxes, bboxes_gt):
    frame_ids_present_gt = np.unique(np.nonzero(~np.isnan(bboxes_gt))[0])
    frame_ids_present = np.unique(np.nonzero(~np.isnan(bboxes))[0])
    frame_ids_present_both = np.intersect1d(frame_ids_present_gt, frame_ids_present, assume_unique=True)
    bboxes_gt_present = bboxes_gt[frame_ids_present_both,0,:]
    bboxes_present = bboxes[frame_ids_present_both,0,:]
    assert np.sum(np.isnan(bboxes_gt_present)) == 0
    assert np.sum(np.isnan(bboxes_present)) == 0
    assert np.sum(bboxes_present[:,0] < bboxes_present[:,2]) == bboxes_present.shape[0]
    assert np.sum(bboxes_present[:,1] < bboxes_present[:,3]) == bboxes_present.shape[0]
    assert np.sum(bboxes_gt_present[:,0] < bboxes_gt_present[:,2]) == bboxes_gt_present.shape[0]
    assert np.sum(bboxes_gt_present[:,1] < bboxes_gt_present[:,3]) == bboxes_gt_present.shape[0]
    bboxes_inter = np.zeros_like(bboxes_present)
    bboxes_inter[:,:2] = np.max(np.concatenate((bboxes_present[:,:2,None],bboxes_gt_present[:,:2,None]),axis=2),axis=2)
    bboxes_inter[:,2:] = np.min(np.concatenate((bboxes_present[:,2:,None],bboxes_gt_present[:,2:,None]),axis=2),axis=2)
    area_inter = (bboxes_inter[:,2] - bboxes_inter[:,0]) * (bboxes_inter[:,3] - bboxes_inter[:,1])
    area_inter[bboxes_inter[:,0] > bboxes_inter[:,2]] = 0
    area_inter[bboxes_inter[:,1] > bboxes_inter[:,3]] = 0
    area1 = (bboxes_present[:,2] - bboxes_present[:,0]) * (bboxes_present[:,3] - bboxes_present[:,1])
    area2 = (bboxes_gt_present[:,2] - bboxes_gt_present[:,0]) * (bboxes_gt_present[:,3] - bboxes_gt_present[:,1])
    area_union = area1 + area2 - area_inter
    sidx = np.nonzero(area_union < 0)
    return frame_ids_present_both, area_inter/area_union

def compute_perc_correct_per_threshold_2D(bboxes, bboxes_gt, predicted, gt, thresholds):
    frame_ids_present_gt = np.unique(np.nonzero(~np.isnan(bboxes_gt))[0])
    frame_ids_present = np.unique(np.nonzero(~np.isnan(bboxes))[0])
    frame_ids_present_both = np.intersect1d(frame_ids_present_gt, frame_ids_present, assume_unique=True)

    gt_present = gt[frame_ids_present_both, ...]
    pred_present = predicted[frame_ids_present_both, ...]
    non_labelled_head_gt = np.sum(np.isnan(gt_present[:,:,:3,2]))
    labelled_head_gt = gt_present[:,:,:3,2].size - non_labelled_head_gt
    non_labelled_body_gt = np.sum(np.isnan(gt_present[:,:,[3,8],2]))
    labelled_body_gt = gt_present[:,:,[3,8],2].size - non_labelled_body_gt
    non_labelled_limbs_gt = np.sum(np.isnan(gt_present[:,:,np.r_[4:8,9:13],2]))
    labelled_limbs_gt = gt_present[:,:,np.r_[4:8,9:13],2].size - non_labelled_limbs_gt
    non_labelled_tail_gt = np.sum(np.isnan(gt_present[:,:,13:,2]))
    labelled_tail_gt = gt_present[:,:,13:,2].size - non_labelled_tail_gt
    error_head = (pred_present[:,:,:3,:2] - gt_present[:,:,:3,:2])
    error_head = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_head,error_head))
    perc_head = [100 * np.sum(error_head < thresh)/labelled_head_gt for thresh in thresholds]

    error_body = (pred_present[:,:,[3,8],:2] - gt_present[:,:,[3,8],:2])
    error_body = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_body,error_body))
    perc_body = [100 * np.sum(error_body < thresh)/labelled_body_gt for thresh in thresholds]

    error_limbs = (pred_present[:,:,np.r_[4:8,9:13],:2] - gt_present[:,:,np.r_[4:8,9:13],:2])
    error_limbs = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_limbs,error_limbs))
    perc_limbs = [100 * np.sum(error_limbs < thresh)/labelled_limbs_gt for thresh in thresholds]

    error_tail = (pred_present[:,:,13:,:2] - gt_present[:,:,13:,:2])
    error_tail = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_tail,error_tail))
    perc_tail = [100 * np.sum(error_tail < thresh)/labelled_tail_gt for thresh in thresholds]

    return perc_head, perc_body, perc_limbs, perc_tail

def compute_perc_correct_per_threshold_3D(groundtruth, predicted, thresholds):
    print(groundtruth[:10,0,:])
    print(predicted[:10,0,:])
    gt_head = groundtruth[:,:3,:]
    non_labelled_head_gt = np.sum(np.isnan(gt_head[:,:,2]))
    labelled_head_gt = gt_head[:,:,2].size - non_labelled_head_gt
    gt_body = groundtruth[:,[3,8],:]
    non_labelled_body_gt = np.sum(np.isnan(gt_body[:,:,2]))
    labelled_body_gt = gt_body[:,:,2].size - non_labelled_body_gt
    gt_limbs = groundtruth[:,np.r_[4:8,9:13],:]
    non_labelled_limbs_gt = np.sum(np.isnan(gt_limbs[:,:,2]))
    labelled_limbs_gt = gt_limbs[:,:,2].size - non_labelled_limbs_gt
    gt_tail = groundtruth[:,13:,:]
    non_labelled_tail_gt = np.sum(np.isnan(gt_tail[:,:,2]))
    labelled_tail_gt = gt_tail[:,:,2].size - non_labelled_tail_gt

    error_head_finetuned = (predicted[:,:3,:] - gt_head)
    error_head_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_head_finetuned,error_head_finetuned))
    perc_head_finetuned = [100 * np.sum(error_head_finetuned < thresh)/labelled_head_gt for thresh in thresholds]

    error_body_finetuned = (predicted[:,[3,8],:] - gt_body)
    error_body_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_body_finetuned,error_body_finetuned))
    perc_body_finetuned = [100 * np.sum(error_body_finetuned < thresh)/labelled_body_gt for thresh in thresholds]

    error_limbs_finetuned = (predicted[:,np.r_[4:8,9:13],:] - gt_limbs)
    error_limbs_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_limbs_finetuned,error_limbs_finetuned))
    perc_limbs_finetuned = [100 * np.sum(error_limbs_finetuned < thresh)/labelled_limbs_gt for thresh in thresholds]

    error_tail_finetuned = (predicted[:,13:,:] - gt_tail)
    error_tail_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_tail_finetuned,error_tail_finetuned))
    perc_tail_finetuned = [100 * np.sum(error_tail_finetuned < thresh)/labelled_tail_gt for thresh in thresholds]

    return perc_head_finetuned, perc_body_finetuned, perc_limbs_finetuned, perc_tail_finetuned

perc_ticks = np.linspace(0,100,5)

fig, axs = plt.subplots(2,2, figsize=(20,15))
fig.subplots_adjust(hspace=0.2)
axs = axs.flatten()
perc_truepositives, perc_falsepositives  = list(zip(*[compute_true_false_positives(bboxes, bboxes_gt) for bboxes, bboxes_gt in zip([bboxes_etho,bboxes_home],[sorted_gt_bboxes_etho,sorted_gt_bboxes_home])]))
scores = ['True positives', 'False positives']
models = ['Cage B', 'Cage K']
width = 1/(len(models) + 1)
offsets = np.arange(width + width/2, 1, width)
xs = np.arange(2)
for i, (tp, fp) in enumerate(zip(perc_truepositives, perc_falsepositives)):
    axs[0].bar(xs + offsets[i], [tp, fp], width, label = models[i],edgecolor = 'black', color = colors[i])
axs[0].set_xticks(xs + 0.5 + width/2, labels=scores, fontsize = tfs)
axs[0].set_yticks(perc_ticks, labels=perc_ticks.astype(int), fontsize = tfs)
axs[0].set_ylabel('Percentage', fontsize=lfs)
axs[0].set_ylim((0,100))
axs[0].legend(prop={'size': lgfs}, frameon=False)
axs[0].set_title('Detection performance',fontsize = fs)
               
idx, ious  = zip(*[compute_iou(bboxes, bboxes_gt) for bboxes, bboxes_gt in zip([bboxes_etho,bboxes_home],[sorted_gt_bboxes_etho,sorted_gt_bboxes_home])])
models = ['Cage B', 'Cage K']
xs = np.arange(1, len(models) + 1)
for i in range(len(models)):
    print(f'{models[i]} IoU: {np.mean(ious[i])}')
    vp = axs[1].violinplot(ious[i:i+1], xs[i:i+1], showmeans=True)
    for k,v in vp.items():
        if k == 'bodies':
            v[0].set_facecolor(colors[i])
        else:
            v.set_color(colors[i])
axs[1].set_xticks(xs, labels=models, fontsize = tfs)
axs[1].set_yticks(np.arange(0,1.01,0.25), labels=np.round(np.arange(0,1.01,0.25),2), fontsize = tfs)
axs[1].set_ylabel('IoU', fontsize = lfs)
axs[1].set_ylim((0,1))
axs[1].set_title('Bbox overlap',fontsize = fs)

thresholds = np.arange(0,70,4)
allbboxes = [bboxes_etho, bboxes_home]
allbboxes_gt = [sorted_gt_bboxes_etho, sorted_gt_bboxes_home]
allpoints = [points_with_score_2d_etho, points_with_score_2d_home]
allpoints_gt = [sorted_gt_keypoints_etho, sorted_gt_keypoints_home]

models = ['B', 'K']

# colors = ['r', 'b','purple']
for i in range(len(models)):
    perc_head, perc_body, perc_limbs, perc_tail = compute_perc_correct_per_threshold_2D(allbboxes[i], allbboxes_gt[i], allpoints[i], allpoints_gt[i], thresholds)
    axs[2].plot(thresholds,perc_head,c=colors[i],marker = 'o',ms=6, label = models[i] + ' head', lw=lw)
    axs[2].plot(thresholds,perc_body,c=colors[i],marker = 's',ms=6, label = models[i] + ' body', lw=lw)
    axs[2].plot(thresholds,perc_limbs,c=colors[i],marker = '^',ms=6, label = models[i] + ' limbs', lw=lw)
    axs[2].plot(thresholds,perc_tail,c=colors[i],marker = 'd',ms=6, label = models[i] + ' tail', lw=lw)
axs[2].legend(prop={'size': lgfs}, frameon=False, ncol=2)
axs[2].set_xlabel('Error threshold (pixels)', fontsize = lfs)
axs[2].set_xticks(np.arange(0,80,20), labels=np.arange(0,80,20), fontsize = tfs)
axs[2].set_ylabel('Accuracy (%)', fontsize = lfs)
axs[2].set_ylim((0,100))
axs[2].set_yticks(perc_ticks, labels=perc_ticks.astype(int), fontsize = tfs)
axs[2].set_title('2D Keypoints detection',fontsize = fs)

thresholds = np.arange(0,100,5)
gt3d = [gt_etho_points_3d, gt_home_points_3d]
points3d = [etho_points_3d, home_points_3d]
for i in range(len(models)):
    perc_head_finetuned, perc_body_finetuned, perc_limbs_finetuned, perc_tail_finetuned = compute_perc_correct_per_threshold_3D(gt3d[i], points3d[i], thresholds)
    axs[3].plot(thresholds,perc_head_finetuned,marker = 'o',ms=6, label = models[i] + ' head',c=colors[i], lw=lw)
    axs[3].plot(thresholds,perc_body_finetuned,marker = 's',ms=6, label = models[i] + ' body',c=colors[i], lw=lw)
    axs[3].plot(thresholds,perc_limbs_finetuned,marker = '^',ms=6, label = models[i] + ' limbs',c=colors[i], lw=lw)
    axs[3].plot(thresholds,perc_tail_finetuned,marker = 'd',ms=6, label = models[i] + ' tail',c=colors[i], lw=lw)
axs[3].set_xlabel('Error threshold (cm)', fontsize = lfs)
axs[3].set_xticks(np.arange(0,101,25), labels=np.arange(0,11,2.5), fontsize = tfs)
axs[3].set_ylabel('Accuracy (%)', fontsize = lfs)
axs[3].set_ylim((0,100))
axs[3].set_yticks(perc_ticks, labels=perc_ticks.astype(int), fontsize = tfs)
axs[3].set_title('3D Keypoints detection',fontsize = fs)

def simpleaxis(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.get_xaxis().tick_bottom()
    ax.get_yaxis().tick_left()
    for spine in ax.spines.values():
        spine.set_linewidth(2)        # thicker axis lines

    # Lengthen and thicken ticks
    ax.tick_params(axis='both', which='major',
                length=10,            # tick length in points
                width=2,              # tick width in points
                direction='out')      # 'in', 'out', or 'inout'

    # If you want longer minor ticks too:
    ax.tick_params(axis='both', which='minor',
                length=6, width=1)

axs = [simpleaxis(ax) for ax in axs]
fig.show()
fig.savefig('Fig2.png', dpi=300)

In [ ]:
print(etho_points_3d[0:10,4,0])
print(gt_etho_points_3d[0:10,4,0])
print(home_points_3d[0:10,4,0])
print(gt_home_points_3d[0:10,4,0])

In [ ]:
print(gt_points_3d[0:10,4,0])
print(finetuned_with_points_3d[0:10,4,0])